In [1]:
# !pip install -U torchao

In [2]:
# !pip install -U bitsandbytes

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import torch

In [4]:
import zipfile

path = '/content/tinyllama-non-instruction.zip'

with zipfile.ZipFile(path, 'r') as zip_ref:
    zip_ref.extractall()
model_path = '/content/checkpoint-5'

In [5]:
# Load dataset
dataset = load_dataset("csv", data_files="/content/pharma_instruction_data.csv", split="train")
print(dataset)


# Format dataset to Alpaca-style text
def format_example(example):
    # Build unified instruction-style prompt
    prompt = f"### Instruction:\n{example['instruction']}\n### Input:\n{example['input']}\n### Response:\n{example['output']}"
    return {"text": prompt}

dataset = dataset.map(format_example)
print(dataset[0]["text"])

base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# Tokenizer setup
tokenizer = AutoTokenizer.from_pretrained(base_model)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# Tokenization with Response Masking
def tokenize_and_mask(example):
    text = example["text"]

    # Tokenize full text
    enc = tokenizer(text, truncation=True, padding="max_length", max_length=512)
    input_ids = enc["input_ids"]

    # Find where '### Response:' starts
    response_marker = "### Response:"
    response_start = text.find(response_marker)

    if response_start != -1:
        # Token index where response begins
        response_token_start = len(tokenizer(text[:response_start])["input_ids"])
    else:
        response_token_start = 0  # if marker not found

    # Clone labels and mask out everything before 'Response'
    labels = input_ids.copy()
    labels[:response_token_start] = [-100] * response_token_start

    enc["labels"] = labels
    return enc

# Apply tokenization
tokenized = dataset.map(tokenize_and_mask, batched=False)
print("Tokenization + masking done.")




# # Load base model (previously non-instructional trained)


# model = get_peft_model(non_instructional_trained_model, lora_config)

# # Training setup
# args = TrainingArguments(
#     output_dir="./tinyllama-instruction",
#     num_train_epochs=30,
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=8,
#     learning_rate=2e-4,
#     fp16=True,
#     logging_steps=15,
#     save_total_limit=1,
#     report_to="none"
# )

# trainer = Trainer(
#     model=model,
#     args=args,
#     train_dataset=tokenized,
# )


# # Train the model
# trainer.train()


# # Save & test the model
# trainer.save_model("/content/tinyllama-instruction")
# tokenizer.save_pretrained("/content/tinyllama-instruction")

# # Test generation
# trained_model = AutoModelForCausalLM.from_pretrained("/content/tinyllama-instruction", device_map="auto")

# prompt = "### Instruction:\nWhat is Ezetimibe?\n### Input:\n\n### Response:\n"
# inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# outputs = trained_model.generate(
#     **inputs,
#     max_new_tokens=100,
#     temperature=0.8,
#     top_p=0.9,
#     do_sample=True,
#     repetition_penalty=1.1
# )

# print("\nModel Output:\n")
# print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 5
})
### Instruction:
Explain the mechanism of action of Metformin.
### Input:
None
### Response:
Metformin activates AMP-activated protein kinase (AMPK), which increases glucose uptake and fatty-acid oxidation while inhibiting hepatic gluconeogenesis, thereby lowering blood glucose.
Tokenization + masking done.


In [12]:
from transformers.utils import quantization_config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=False,
)

# Load the base model without quantization first for merging purposes
model = AutoModelForCausalLM.from_pretrained(base_model, quantization_config = bnb_config, device_map='auto')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [13]:
model = PeftModel.from_pretrained(model, model_path)
model = model.merge_and_unload()

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [14]:
# LoRA config
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [15]:
model = get_peft_model(model, lora_config)

# Training setup
args = TrainingArguments(
    output_dir="./tinyllama-instruction",
    num_train_epochs=30,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
)

# Train the model
trainer.train()

Step,Training Loss
10,13.556175
20,4.468679
30,1.508523


TrainOutput(global_step=30, training_loss=6.511125946044922, metrics={'train_runtime': 64.9227, 'train_samples_per_second': 2.31, 'train_steps_per_second': 0.462, 'total_flos': 477222351667200.0, 'train_loss': 6.511125946044922, 'epoch': 30.0})

In [19]:
# Save & test the model
trainer.save_model("/content/tinyllama-instruction")
tokenizer.save_pretrained("/content/tinyllama-instruction")

# Test generation
trained_model = AutoModelForCausalLM.from_pretrained("/content/tinyllama-instruction", device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [16]:
#non_instructional domain specific model
non_instructional_trained_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [17]:
question = "Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry."

In [21]:
inputs = tokenizer(question, return_tensors="pt").to("cuda")

outputs = non_instructional_trained_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    )
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry.
A.I. is a key enabler for drug discovery and development. In the past, scientists developed drugs by trial and error. Today, scientists use sophisticated software to analyze the properties of drugs and predict the potential for the drug to work in certain cells. In the future, AI will also be able to predict how a drug will be metabolized in the body and the potential side effects. AI will also be able to recommend a new


In [20]:
prompt = f"### Instruction:\n{question}\n### Input:\n{question}\n### Response:\n"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = trained_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.9,
    top_p=0.8,
    do_sample=True,
    repetition_penalty=1.1
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Instruction:
Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry.
### Input:
Explain how artificial intelligence is improving the process of drug discovery and development in the pharmaceutical industry.
### Response:
Artificial intelligence (AI) has transformed the drug discovery and development processes by providing a scalable, cost-effective, and efficient alternative to traditional manual methods. AI algorithms are designed to analyze vast amounts of data generated during the drug discovery process, such as genetic sequences, chemical structures, and molecular properties, to identify promising leads or candidates for drug development. These algorithms use natural language processing (NLP), machine learning (ML), and deep learning (DL) technologies
